In [1]:
print("Downloading files from public repository...")

# The -f flag tells curl to fail on an error (like 404)
!curl -Lf https://raw.githubusercontent.com/Denis2054/Context-Engineering-for-Multi-Agent-Systems/main/commons/utils.py --output utils.py
!curl -Lf https://raw.githubusercontent.com/Denis2054/Context-Engineering-for-Multi-Agent-Systems/main/commons/ch8/helpers.py --output helpers.py
!curl -Lf https://raw.githubusercontent.com/Denis2054/Context-Engineering-for-Multi-Agent-Systems/main/commons/ch8/agents.py --output agents.py
!curl -Lf https://raw.githubusercontent.com/Denis2054/Context-Engineering-for-Multi-Agent-Systems/main/commons/ch8/registry.py --output registry.py
!curl -Lf https://raw.githubusercontent.com/Denis2054/Context-Engineering-for-Multi-Agent-Systems/main/commons/ch8/engine.py --output engine.py
# (You might want to add a check here to see if the files actually exist now)
print("✅ File download attempt finished!")

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100  1823  100  1823    0     0   5908      0 --:--:-- --:--:-- --:--:--  5899
  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100  6479  100  6479    0     0  19904      0 --:--:-- --:--:-- --:--:-- 19935
  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100  9440  100  9440    0     0  29261      0 --:--:-- --:--:-- --:--:-- 29316
  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100  3611  100  3611    0     0  11665      0 --:--:-- --:--:-- --:--:-- 11648
  % Total    % Received % Xferd  Average Speed   Tim

In [2]:
#Installation and Client Setup

# Import the setup functions from your new utility file
import utils

# Run the installation
utils.install_dependencies()

# Initialize the OpenAI and Pinecone clients
client, pc = utils.initialize_clients()

🚀 Installing required packages...
✅ All packages installed successfully.

🔑 Initializing API clients...
   - OpenAI client initialized.
   - Pinecone client initialized.
✅ Clients initialized successfully.


In [3]:
# 1. Import the hardened helper functions (LLM, Embeddings, Pinecone)
import helpers

# 2. Import the specialist agent functions (Librarian, Researcher, Writer)
import agents

# 3. Import the AGENT_TOOLKIT object that knows about all the agents
from registry import AGENT_TOOLKIT

# 4. Import the main context_engine function that orchestrates the entire process
from engine import context_engine

In [4]:
import json
import html
import markdown
from IPython.display import display, HTML

def render_trace_dashboard(trace):
    """
    Generates a clean HTML dashboard for the Context Engine Execution Trace.
    Smartly unpacks JSON outputs to render the Markdown text inside them.
    """
    # Define CSS styles for the dashboard
    css = """

    """

    # Determine Status
    status_class = "status-success" if trace.status == "Success" else "status-failure"

    # Header
    dashboard_html = f"""
    {css}



                Context Engine Trace
                "{html.escape(trace.goal)}"


                {trace.status}

                    Time: {trace.duration:.2f}s





            Execution Steps
    """

    # Loop through steps
    for step in trace.steps:
        # 1. Handle Input (Just JSON dump usually)
        try:
            resolved_ctx = json.dumps(step['resolved_context'], indent=2)
        except Exception:
            resolved_ctx = str(step['resolved_context'])

        # 2. Handle Output (Smart Unpacking)
        output_raw = step['output']
        rendered_html = ""
        raw_json_display = ""

        # Helper to render text as markdown
        def render_md(text):
            if not text: return ""
            try:
                return markdown.markdown(text)
            except:
                return html.escape(text)

        if isinstance(output_raw, dict):
            # It's a dict, let's look for rich content keys
            rich_keys = ['summary', 'answer_with_sources', 'answer', 'output', 'content', 'blueprint_json']
            found_content = None

            for key in rich_keys:
                if key in output_raw and isinstance(output_raw[key], str):
                    found_content = output_raw[key]
                    # If it looks like a JSON string (blueprint), format it as code, otherwise markdown
                    if found_content.strip().startswith('{') and found_content.strip().endswith('}'):
                         try:
                             parsed = json.loads(found_content)
                             pretty = json.dumps(parsed, indent=2)
                             rendered_html = f'{html.escape(pretty)}'
                         except:
                             rendered_html = f'{render_md(found_content)}'
                    else:
                        rendered_html = f'{render_md(found_content)}'
                    break

            # Prepare the raw JSON as a backup/toggle
            raw_json_display = json.dumps(output_raw, indent=2)

        elif isinstance(output_raw, str):
            # Direct string output (usually final writer step)
            rendered_html = f'{render_md(output_raw)}'
            raw_json_display = output_raw # In this case "raw" is just the string
        else:
            raw_json_display = str(output_raw)

        # Build Step HTML
        step_html = f"""



                        Step {step['step']}
                        {step['agent']}

                    ▼ Toggle



                        Input (Resolved)

                            View Input JSON
                            {html.escape(resolved_ctx)}



                        Output
                        {rendered_html}


                            View Raw Output Data
                            {html.escape(raw_json_display)}




        """
        dashboard_html += step_html

    dashboard_html += "" # Close steps container

    # Final Output Section
    if trace.final_output:
        # Try to find the best string to render
        content_to_render = trace.final_output
        if isinstance(trace.final_output, dict):
             # Try specific keys first
             for key in ['summary', 'answer_with_sources', 'answer', 'output']:
                 if key in trace.final_output:
                     content_to_render = trace.final_output[key]
                     break
             # If still dict, dump it
             if isinstance(content_to_render, dict):
                 content_to_render = json.dumps(content_to_render, indent=2)

        # Render
        if isinstance(content_to_render, str):
            final_html = markdown.markdown(content_to_render)
        else:
            final_html = f"{html.escape(str(content_to_render))}"

        dashboard_html += f"""

            FINAL RESULT

               {final_html}


        """

    dashboard_html += ""

    display(HTML(dashboard_html))

In [5]:
# === ENGINE ROOM: The Main Execution Function ===
# This function contains all the logic to run the engine.
# We define it here so our final cell can be very simple.

import logging
import pprint
import json  # <--- Added for type handling
from IPython.display import display, Markdown

# === ENGINE ROOM: The Main Execution Function (Visualizer Update) ===

import logging
import pprint
import json
from IPython.display import display, Markdown

def execute_and_display(goal, config, client, pc, moderation_active):
    """
    Runs the context engine with HTML dashboard visualization.
    """
    # --- PRE-FLIGHT MODERATION CHECK (on user input) ---\n
    if moderation_active:
        print("--- [Safety Guardrail] Performing Pre-Flight Moderation Check on Goal ---")
        moderation_report = helpers.helper_moderate_content(text_to_moderate=goal, client=client)

        if moderation_report["flagged"]:
            print("\n🛑 Goal failed pre-flight moderation. Execution halted.")
            pprint.pprint(moderation_report)
            return

    logging.info(f"******** Starting Engine for Goal: '{goal}' **********\n")

    # 1. Run the Context Engine
    result, trace = context_engine(
        goal,
        client=client,
        pc=pc,
        **config
    )

    # --- POST-FLIGHT MODERATION CHECK (on AI output) ---\n
    if result and moderation_active:
        # Flatten result for checking
        text_to_check = str(result)
        if isinstance(result, (dict, list)):
            text_to_check = json.dumps(result)

        moderation_report = helpers.helper_moderate_content(text_to_moderate=text_to_check, client=client)

        if moderation_report["flagged"]:
            print("\n🛑 Generated output failed post-flight moderation and will be redacted.")
            result = "[Content flagged as potentially harmful by moderation policy and has been redacted.]"
            # We also scrub the trace result to prevent display of harmful content
            trace.final_output = result

    # 2. Render the HTML Dashboard
    # This replaces the raw text printouts with the visualizer
    if trace:
        render_trace_dashboard(trace)
    else:
        print("Engine failed to initialize trace.")


In [7]:
# 1. Define all configuration variables for this run in a dictionary
config = {
    "index_name": 'genai-mas-mcp-ch3',
    "generation_model": "gpt-5.1",
    "embedding_model": "text-embedding-3-small",
    "namespace_context": 'ContextLibrary',
    "namespace_knowledge": 'KnowledgeStore'
}



In [8]:
#@title CONTROL DECK: Moderation
# 1. Define a simple, safe goal to test the moderation workflow.
goal = "Summarize the key points of the QuantumDrive"

# 2. Define the standard configuration.
config = {
    "index_name": 'genai-mas-mcp-ch3',
    "generation_model": "gpt-5.1",
    "embedding_model": "text-embedding-3-small",
    "namespace_context": 'ContextLibrary',
    "namespace_knowledge": 'KnowledgeStore'
}

# 3. Call the execution function with moderation explicitly activated.
execute_and_display(goal, config, client, pc, moderation_active=True)


--- [Safety Guardrail] Performing Pre-Flight Moderation Check on Goal ---


ERROR:root:OpenAI API Error in call_llm_robust: Error code: 429 - {'error': {'message': 'You have no credits remaining. Add credits to continue using the API at https://platform.openai.com/settings/organization/billing/.', 'type': 'insufficient_quota', 'param': None, 'code': 'credit_balance_exhausted'}}
ERROR:root:OpenAI API Error in call_llm_robust: Error code: 429 - {'error': {'message': 'You have no credits remaining. Add credits to continue using the API at https://platform.openai.com/settings/organization/billing/.', 'type': 'insufficient_quota', 'param': None, 'code': 'credit_balance_exhausted'}}
ERROR:root:OpenAI API Error in call_llm_robust: Error code: 429 - {'error': {'message': 'You have no credits remaining. Add credits to continue using the API at https://platform.openai.com/settings/organization/billing/.', 'type': 'insufficient_quota', 'param': None, 'code': 'credit_balance_exhausted'}}
ERROR:root:OpenAI API Error in call_llm_robust: Error code: 429 - {'error': {'message

In [9]:
#@title Product Marketing Copy Generation(Use Case 1)
goal = "Analyze the ChronoTech press release and summarize their core product messaging and value proposition. Please cite your sources."


# 2. Use the standard configuration
config = {
    "index_name": 'genai-mas-mcp-ch3',
    "generation_model": "gpt-5.1", # or your preferred model
    "embedding_model": "text-embedding-3-small",
    "namespace_context": 'ContextLibrary',
    "namespace_knowledge": 'KnowledgeStore'
}

# 3. Call the execution function
execute_and_display(goal, config, client, pc,moderation_active=False)

ERROR:root:OpenAI API Error in call_llm_robust: Error code: 429 - {'error': {'message': 'You have no credits remaining. Add credits to continue using the API at https://platform.openai.com/settings/organization/billing/.', 'type': 'insufficient_quota', 'param': None, 'code': 'credit_balance_exhausted'}}
ERROR:root:OpenAI API Error in call_llm_robust: Error code: 429 - {'error': {'message': 'You have no credits remaining. Add credits to continue using the API at https://platform.openai.com/settings/organization/billing/.', 'type': 'insufficient_quota', 'param': None, 'code': 'credit_balance_exhausted'}}
ERROR:root:OpenAI API Error in call_llm_robust: Error code: 429 - {'error': {'message': 'You have no credits remaining. Add credits to continue using the API at https://platform.openai.com/settings/organization/billing/.', 'type': 'insufficient_quota', 'param': None, 'code': 'credit_balance_exhausted'}}
ERROR:root:OpenAI API Error in call_llm_robust: Error code: 429 - {'error': {'message

In [10]:
#@title Product Marketing Copy Generation(Use Case 2)


# 1. Define the Goal: A research query that requires a verifiable, cited answer.
#    - DOMAIN: Any knowledge-intensive field (e.g., legal, medical, financial).
#    - KEY CAPABILITY: Tests the high-fidelity `Researcher` agent and its ability
#      to retrieve text with `source` metadata and generate citations.
# goal = "[INSERT YOUR HIGH-FIDELITY RESEARCH GOAL HERE]"

## 1. Define the Goal: A research query that asks for a creative output.
goal = "Using the official product spec sheet, write a short marketing description for the new QuantumDrive Q-1. The description should be confident, aspirational, and focus on the benefits for creative professionals. Please cite your sources."



# 2. Use the standard configuration
config = {
    "index_name": 'genai-mas-mcp-ch3',
    "generation_model": "gpt-5.1", # or your preferred model
    "embedding_model": "text-embedding-3-small",
    "namespace_context": 'ContextLibrary',
    "namespace_knowledge": 'KnowledgeStore'
}

# 3. Call the execution function
execute_and_display(goal, config, client, pc,moderation_active=False)

ERROR:root:OpenAI API Error in call_llm_robust: Error code: 429 - {'error': {'message': 'You have no credits remaining. Add credits to continue using the API at https://platform.openai.com/settings/organization/billing/.', 'type': 'insufficient_quota', 'param': None, 'code': 'credit_balance_exhausted'}}
ERROR:root:OpenAI API Error in call_llm_robust: Error code: 429 - {'error': {'message': 'You have no credits remaining. Add credits to continue using the API at https://platform.openai.com/settings/organization/billing/.', 'type': 'insufficient_quota', 'param': None, 'code': 'credit_balance_exhausted'}}
ERROR:root:OpenAI API Error in call_llm_robust: Error code: 429 - {'error': {'message': 'You have no credits remaining. Add credits to continue using the API at https://platform.openai.com/settings/organization/billing/.', 'type': 'insufficient_quota', 'param': None, 'code': 'credit_balance_exhausted'}}
ERROR:root:OpenAI API Error in call_llm_robust: Error code: 429 - {'error': {'message

In [11]:
#@title Writing a brand pitch recommendation(Use case 3)

# 1. Define the Goal: A creative or factual task that is deliberately
#    outside the scope of the documents in the knowledge base.
#    - DOMAIN: Universal test applicable to any curated knowledge base.
#    - KEY CAPABILITY: Tests the `Researcher` agent's ability to report a
#      negative finding and the `Writer` agent's ability to handle it gracefully,
#      preventing hallucination.
# goal = "[INSERT YOUR OUT-OF-SCOPE GOAL HERE]"


# =CONTROL DECK 3: A persuasive pitch ===
goal = "Write a persuasive pitch on our brand tone and voice guide"


# 2. Use the same configuration dictionary
config = {
    "index_name": 'genai-mas-mcp-ch3',
    "generation_model": "gpt-5.1", # or your preferred model
    "embedding_model": "text-embedding-3-small",
    "namespace_context": 'ContextLibrary',
    "namespace_knowledge": 'KnowledgeStore'
}

# 3. Call the execution function
execute_and_display(goal, config, client, pc,moderation_active=False)

ERROR:root:OpenAI API Error in call_llm_robust: Error code: 429 - {'error': {'message': 'You have no credits remaining. Add credits to continue using the API at https://platform.openai.com/settings/organization/billing/.', 'type': 'insufficient_quota', 'param': None, 'code': 'credit_balance_exhausted'}}
ERROR:root:OpenAI API Error in call_llm_robust: Error code: 429 - {'error': {'message': 'You have no credits remaining. Add credits to continue using the API at https://platform.openai.com/settings/organization/billing/.', 'type': 'insufficient_quota', 'param': None, 'code': 'credit_balance_exhausted'}}
ERROR:root:OpenAI API Error in call_llm_robust: Error code: 429 - {'error': {'message': 'You have no credits remaining. Add credits to continue using the API at https://platform.openai.com/settings/organization/billing/.', 'type': 'insufficient_quota', 'param': None, 'code': 'credit_balance_exhausted'}}
ERROR:root:OpenAI API Error in call_llm_robust: Error code: 429 - {'error': {'message